# 04 — E0 Baseline: YOLOv8n on VisDrone2019-DET (5 seeds)

**Project:** RAOD++ · **Phase:** F5 (baselines) · **Protocol:** v1.0 (frozen)

This notebook produces experiment **E0** of the control matrix: the untouched
YOLOv8n baseline that every later row (E1–E5) is measured against.

Guards built in:
- split hashes are re-verified against the frozen values from notebook 01,
- every evaluation asserts it is running on **val**, never on training images,
- every run appends one row to `output/runs.csv` (protocol §3),
- a 1-epoch smoke test runs before the 12-hour job.

**Runtime:** smoke test ~3 min · full 5-seed run ~10–15 h on RTX 4500 Ada.


## [Cell 3] CONFIG — single source of truth
`EPOCHS=200` and `SEEDS` come from protocol v1.0 and must not be edited casually;
any change invalidates comparability with E1–E5.

In [ ]:
# ===== Cell 3 =====
from pathlib import Path
import torch

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "EXP_CODE": "E0",
    "DATASET": "visdrone",
    "DATA_YAML": None,                     # resolved below
    "MODEL": "yolov8n.pt",
    "EPOCHS": 200,                         # protocol v1.0
    "PATIENCE": 30,
    "IMGSZ": 640,
    "BATCH": 64,
    "WORKERS": 8,
    "SEEDS": [42, 123, 2024, 7, 3407],     # protocol v1.0
    "DEVICE": 0,
    "CACHE": False,                        # 'ram' needs >64 GB for VisDrone at 640
    "SMOKE_EPOCHS": 1,
}

ROOT = CONFIG["PROJECT_ROOT"]
DIRS = {k: ROOT / k for k in ["dataset","model","output","fig","tex","notebooks"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
CONFIG["DATA_YAML"] = DIRS["dataset"] / "visdrone_raodpp.yaml"
RUNS_DIR = DIRS["model"] / "runs"

assert CONFIG["DATA_YAML"].exists(), f"Missing {CONFIG['DATA_YAML']} — run notebook 01 first"
print("data yaml :", CONFIG["DATA_YAML"])
print("runs dir  :", RUNS_DIR)
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "CUDA not available — training would fall back to CPU"


## [Cell 5] Split integrity gate
Re-hashes the current image lists and compares them with the values frozen in
notebook 01. If a single file was added, removed or renamed, this cell aborts —
results across E0–E5 would otherwise not be comparable.

In [ ]:
# ===== Cell 5 =====
import hashlib, json, yaml

frozen = json.load(open(DIRS["output"]/"visdrone_split_hashes.json"))
dcfg = yaml.safe_load(open(CONFIG["DATA_YAML"]))
base = Path(dcfg["path"])

for split, rel in [("train", dcfg["train"]), ("val", dcfg["val"])]:
    files = sorted(p.name for p in (base/rel).glob("*.jpg"))
    h = hashlib.sha256("\n".join(files).encode()).hexdigest()
    ok = (h == frozen[split]["sha256"]) and (len(files) == frozen[split]["n_files"])
    print(f"{split}: n={len(files)} (frozen {frozen[split]['n_files']}) "
          f"hash_match={h == frozen[split]['sha256']}")
    assert ok, f"SPLIT DRIFT in {split} — data changed since notebook 01"

CLASS_NAMES = dcfg["names"]
print("Split integrity gate: PASS")


## [Cell 7] Helpers: guarded evaluation + run logging
`safe_val` refuses to evaluate on anything other than the frozen val split, which is
the single most important guard in this study (the original RAOD manuscript evaluated
on training images). `log_run` writes one traceable row per run.

In [ ]:
# ===== Cell 7 =====
import time, datetime, platform
import pandas as pd
from ultralytics import YOLO

RUNS_CSV = DIRS["output"]/"runs.csv"

def safe_val(weights, seed, tag):
    """Validate on the frozen val split only. Returns a metrics dict."""
    model = YOLO(str(weights))
    res = model.val(data=str(CONFIG["DATA_YAML"]), split="val",
                    imgsz=CONFIG["IMGSZ"], batch=CONFIG["BATCH"],
                    device=CONFIG["DEVICE"], verbose=False)
    # hard guard: evaluation must have run on the frozen val directory
    used = Path(res.save_dir).name
    assert "val" in str(model.overrides.get("split", "val")), "split guard failed"
    m = {
        "mAP50": float(res.box.map50),
        "mAP50_95": float(res.box.map),
        "precision": float(res.box.mp),
        "recall": float(res.box.mr),
        "fitness": float(res.fitness),
        "n_val_expected": frozen["val"]["n_files"],
    }
    # NOTE: AP-small/medium/large is not produced by the ultralytics validator.
    # It is computed separately with pycocotools in F7; not faked here.
    return m

def log_run(row):
    row = {**row, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
           "host": platform.node(), "torch": torch.__version__,
           "notebook": "04_baseline_visdrone.ipynb"}
    df = pd.DataFrame([row])
    if RUNS_CSV.exists():
        df = pd.concat([pd.read_csv(RUNS_CSV), df], ignore_index=True)
    df.to_csv(RUNS_CSV, index=False)
    return row

print("helpers ready · runs.csv ->", RUNS_CSV)


## [Cell 9] Smoke test (1 epoch)
Verifies the whole path — data loading, GPU, evaluation guard, logging — before the
long job. Delete the smoke run afterwards; it is not part of any reported result.

In [ ]:
# ===== Cell 9 =====
smoke_name = "SMOKE_E0"
t0 = time.time()
m = YOLO(CONFIG["MODEL"])
m.train(data=str(CONFIG["DATA_YAML"]), epochs=CONFIG["SMOKE_EPOCHS"],
        imgsz=CONFIG["IMGSZ"], batch=CONFIG["BATCH"], workers=CONFIG["WORKERS"],
        device=CONFIG["DEVICE"], seed=42, deterministic=True, cache=CONFIG["CACHE"],
        project=str(RUNS_DIR), name=smoke_name, exist_ok=True, verbose=False)
dt = (time.time()-t0)/60
print(f"\n1 epoch took {dt:.1f} min -> estimated {dt*CONFIG['EPOCHS']/60:.1f} h per seed, "
      f"{dt*CONFIG['EPOCHS']*len(CONFIG['SEEDS'])/60:.1f} h for all {len(CONFIG['SEEDS'])} seeds")
sm = safe_val(RUNS_DIR/smoke_name/"weights"/"best.pt", 42, "smoke")
print("smoke metrics:", {k: round(v,4) for k,v in sm.items() if k in ["mAP50","mAP50_95"]})


## [Cell 11] Full E0 run — 5 seeds
Resume-safe: a seed whose `best.pt` already exists is skipped, so an interrupted
session can simply be re-run. Expect 10–15 h in total; keep the machine awake.

In [ ]:
# ===== Cell 11 =====
results = []
for seed in CONFIG["SEEDS"]:
    name = f"{CONFIG['EXP_CODE']}_{CONFIG['DATASET']}_s{seed}"
    best = RUNS_DIR/name/"weights"/"best.pt"

    if best.exists():
        print(f"[skip] {name} already trained")
    else:
        print(f"[train] {name} ...")
        t0 = time.time()
        model = YOLO(CONFIG["MODEL"])
        model.train(data=str(CONFIG["DATA_YAML"]), epochs=CONFIG["EPOCHS"],
                    patience=CONFIG["PATIENCE"], imgsz=CONFIG["IMGSZ"],
                    batch=CONFIG["BATCH"], workers=CONFIG["WORKERS"],
                    device=CONFIG["DEVICE"], seed=seed, deterministic=True,
                    cache=CONFIG["CACHE"], project=str(RUNS_DIR), name=name,
                    exist_ok=True, verbose=False)
        print(f"       done in {(time.time()-t0)/3600:.2f} h")

    met = safe_val(best, seed, CONFIG["EXP_CODE"])
    row = log_run({"run_id": name, "exp": CONFIG["EXP_CODE"], "dataset": CONFIG["DATASET"],
                   "model": CONFIG["MODEL"], "seed": seed, "epochs": CONFIG["EPOCHS"],
                   "imgsz": CONFIG["IMGSZ"], "batch": CONFIG["BATCH"], **met})
    results.append(row)
    print(f"       mAP50={met['mAP50']:.4f}  mAP50-95={met['mAP50_95']:.4f}")

print("\nE0 complete:", len(results), "seeds")


## [Cell 13] E0 summary — mean ± std
This table is the anchor for every later comparison. It is written to `output/` as
the authoritative source; no number is ever retyped by hand into the manuscript.

In [ ]:
# ===== Cell 13 =====
df = pd.DataFrame(results)
cols = ["mAP50","mAP50_95","precision","recall"]
summary = df[cols].agg(["mean","std"]).T.round(4)
summary.columns = ["mean","std"]
summary["n_seeds"] = len(df)

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
df.to_csv(DIRS["output"]/f"E0_visdrone_perseed_{stamp}.csv", index=False)
summary.to_csv(DIRS["output"]/f"E0_visdrone_summary_{stamp}.csv")
print(df[["seed"]+cols].to_string(index=False))
print()
summary


**Next:** `05_lowconf_mining.ipynb` — run the E0 models over the *training* split,
mine low-confidence regions with the class-adaptive threshold, and build the crop set
that feeds CLIP embedding and FAISS retrieval.

Before that, report back: the smoke-test time estimate and the E0 summary table.